# The Chain-of-Agents Orchestrator

**Workshop Agent 3** — *10 Essential AI Agents Every Engineer Must Build* Workshop  
**Book:** *30 Agents Every AI Engineer Must Build* by Imran Ahmad (Packt Publishing, 2026)  
**Chapter Reference:** Chapter 7 — Sections 7.4–7.6 and 7.7b (pp. 186–194, 198–201)

---

In this workshop agent you will build a **multi-agent insurance-claims workflow governed by a state machine with human-in-the-loop escalation**. You begin with the Chain-of-Agents pattern itself: a manager agent coordinates a team of specialist agents under a four-pillar **Cooperation Protocol**, records their findings in shared **episodic memory**, and detects disagreement between their outputs with a calibrated **conflict score**. You then apply the same orchestration principles to an insurance claims pipeline — intake with OCR-style field extraction, policy validation, risk classification, and payout — modeled as a state machine with explicit guard conditions. In the final demo you will trace three test claims end-to-end: the confident path is auto-approved, a low-confidence claim is escalated to a human reviewer, and an invalid claim is rejected at validation — with every state transition written to a **full audit trail**.

The notebook runs end-to-end in **Simulation Mode**; no API key is required. Sections covered: §7.4 (the Cooperation Protocol), §7.5 (Memory-Augmented Multi-Agent Systems), §7.6 (Conflict Resolution Mechanisms), and §7.7b (Agentic Workflow — Insurance Claims Processing). The section numbering below follows the original chapter notebook; Sections 1–3 (the Tool-Using Agent) and Section 6 (the e-commerce workflow) are covered elsewhere, and Section 7 runs standalone without them.


In [ ]:
# Google Colab bootstrap — runs only on Colab, no-op everywhere else.
# Locally you are already inside the agent folder with requirements installed.
import os
import sys

if "google.colab" in sys.modules:
    AGENT_DIR = "03-chain-of-agents-orchestrator"
    if not os.path.exists("/content/repo"):
        os.system("git clone --depth 1 https://github.com/cloudanum/ws-10-agents /content/repo")
    os.chdir(f"/content/repo/{AGENT_DIR}")
    # On Colab, prefer requirements-colab.txt when present: it drops pins that
    # cannot coexist with Colab's preinstalled stack (e.g. langchain 0.2.16
    # requires numpy<2 on Python 3.13, while Colab ships numpy 2.x).
    req_file = "requirements-colab.txt" if os.path.exists("requirements-colab.txt") else "requirements.txt"
    # Keep Colab's preinstalled scientific/kernel stack: the kernel already has
    # numpy, pandas, pydantic and ipykernel loaded, so letting pip replace them
    # (e.g. building numpy 1.26.4 from source or upgrading ipykernel) breaks the
    # running kernel with ABI errors or an OOM kill. Filter those lines out of
    # requirements and constrain the rest of the install to the installed versions.
    import re
    from importlib.metadata import PackageNotFoundError, version
    filtered = [
        line for line in open(req_file)
        if not re.match(r"\s*(numpy|pandas|pydantic|jupyter|ipykernel)\b", line, re.IGNORECASE)
    ]
    with open("/tmp/colab_requirements.txt", "w") as fh:
        fh.writelines(filtered)
    pins = []
    for pkg in ("numpy", "pandas", "pydantic", "ipykernel"):
        try:
            pins.append(f"{pkg}=={version(pkg)}")
        except PackageNotFoundError:
            pass
    with open("/tmp/colab_constraints.txt", "w") as fh:
        fh.write("\n".join(pins) + "\n")
    import subprocess
    res = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "-r", "/tmp/colab_requirements.txt",
         "--constraint", "/tmp/colab_constraints.txt"],
        capture_output=True, text=True,
    )
    if res.returncode != 0:
        print("pip install failed — re-running without -q for the full resolver report:\n")
        subprocess.run(
            [sys.executable, "-m", "pip", "install",
             "-r", "/tmp/colab_requirements.txt",
             "--constraint", "/tmp/colab_constraints.txt"],
        )
        raise RuntimeError("Colab bootstrap: pip install failed (see resolver report above)")
    print(f"Colab setup complete ({req_file}) — working directory: {os.getcwd()}")
else:
    print("Not on Colab — skipping bootstrap (local setup already in place).")

---
## Section 0: Setup and Configuration

This section loads dependencies, detects the operating mode (Simulation vs. Live),
and initializes the shared infrastructure used by all subsequent sections.

**Key components initialized here:**
- `helpers` package (color logger, resilience decorators, MockLLM)
- `LIVE_MODE` and `INTERACTIVE_MODE` flags
- `AgentError` exception class used by workflow sections

In [1]:
# =============================================================================
# Section 0: Setup and Configuration
# Chapter 7: Tool Manipulation and Orchestration Agents
# Book: "Agents" by Imran Ahmad (Packt, 2026 — B34135)
# =============================================================================

import os
import sys
import json
import time
import getpass
from typing import Dict, Any, List, Tuple, Optional

import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

# Ensure helpers/ is importable from the notebook's working directory
sys.path.insert(0, os.path.abspath("."))

from helpers import (
    log_info, log_success, log_error, log_warning, log_mock, log_step,
    graceful_fallback, safe_invoke,
    MockLLM,
)

# Matplotlib backend for inline rendering
%matplotlib inline

# ---------------------------------------------------------------------------
# Custom Exception — used by workflow sections (Sections 7.7, 7.7b)
# ---------------------------------------------------------------------------
class AgentError(Exception):
    """Raised when a workflow step fails critically and the pipeline must stop."""
    pass

log_info("Core imports complete. helpers package loaded.")

[INFO] Core imports complete. helpers package loaded.


In [2]:
# Multi-provider LLM support (OpenAI / Anthropic / Google Gemini)
# Set LLM_PROVIDER in .env to choose: openai | anthropic | google | auto
# Auto-detection uses the first available key.
# See supporting/llm_provider.py for details.

import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))
sys.path.insert(0, '..')

try:
    from supporting.llm_provider import detect_provider, get_llm, PROVIDER_MODELS, print_provider_banner
    _PROVIDER, _PROVIDER_KEY, _PROVIDER_MODE = detect_provider()
    print_provider_banner(_PROVIDER, _PROVIDER_MODE)
except ImportError:
    print('[INFO] supporting/llm_provider.py not found — using default OpenAI path')
    _PROVIDER, _PROVIDER_KEY, _PROVIDER_MODE = 'openai', os.getenv('OPENAI_API_KEY'), 'LIVE' if os.getenv('OPENAI_API_KEY') else 'SIMULATION'



   SIMULATION MODE ACTIVE
   Using MockLLM — no API key required



In [3]:
# =============================================================================
# Section 0: API Key Detection and Mode Selection
# Ref: Strategy §3.4 — Simulation Mode Activation Flow
# =============================================================================

# --- Step 1: Try .env file ---
try:
    from dotenv import load_dotenv
    load_dotenv()
    log_info("python-dotenv loaded. Checking .env for OPENAI_API_KEY...")
except ImportError:
    log_warning("python-dotenv not installed. Skipping .env loading.")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()

# --- Step 2: Interactive fallback via getpass ---
# Set False for Colab, CI/CD, nbconvert --execute, or non-interactive envs
INTERACTIVE_MODE = sys.stdin.isatty() if hasattr(sys.stdin, "isatty") else False

if not OPENAI_API_KEY and INTERACTIVE_MODE:
    try:
        OPENAI_API_KEY = getpass.getpass(
            "Enter OpenAI API key (or press Enter to skip): "
        ).strip()
    except (EOFError, OSError, Exception):
        # Catches StdinNotImplementedError, EOFError, and any other input failure
        OPENAI_API_KEY = ""
        INTERACTIVE_MODE = False
        log_warning("Non-interactive environment detected. INTERACTIVE_MODE set to False.")

# --- Step 3: Determine operating mode ---
LIVE_MODE = bool(OPENAI_API_KEY) and "your-key" not in OPENAI_API_KEY and "your_key" not in OPENAI_API_KEY

if LIVE_MODE:
    log_success("LIVE MODE active. LLM calls will use the OpenAI API.")
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    try:
        from openai import OpenAI
        _openai_client = OpenAI(api_key=OPENAI_API_KEY)
        class LiveLLM:
            """Thin wrapper around OpenAI client matching MockLLM.generate() interface."""
            def generate(self, prompt, **kwargs):
                response = _openai_client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.7,
                )
                return response.choices[0].message.content
        llm = LiveLLM()
    except Exception as e:
        log_error(f"OpenAI client init failed: {e}. Falling back to Simulation.")
        LIVE_MODE = False
        llm = MockLLM(verbose=True)
else:
    llm = MockLLM(verbose=True)
    log_mock("╔══════════════════════════════════════════════════════════╗")
    log_mock("║  SIMULATION MODE ACTIVE                                 ║")
    log_mock("║  All LLM calls return chapter-derived mock responses.   ║")
    log_mock("║  To use a live API, create .env from .env.template      ║")
    log_mock("╚══════════════════════════════════════════════════════════╝")

# --- Summary ---
log_info(f"Operating Mode: {'LIVE' if LIVE_MODE else 'SIMULATION'}")
log_info(f"Interactive Mode: {INTERACTIVE_MODE}")
log_info(f"Python: {sys.version.split()[0]}")
log_info(f"pandas: {pd.__version__}, matplotlib: {matplotlib.__version__}")

[INFO] python-dotenv loaded. Checking .env for OPENAI_API_KEY...
[MOCK] ╔══════════════════════════════════════════════════════════╗
[MOCK] ║  SIMULATION MODE ACTIVE                                 ║
[MOCK] ║  All LLM calls return chapter-derived mock responses.   ║
[MOCK] ║  To use a live API, create .env from .env.template      ║
[MOCK] ╚══════════════════════════════════════════════════════════╝
[INFO] Operating Mode: SIMULATION
[INFO] Interactive Mode: False
[INFO] Python: 3.14.5
[INFO] pandas: 3.0.5, matplotlib: 3.11.1


---
## Section 4: The Chain-of-Agents Orchestrator

**Chapter Reference:** Sections 7.4–7.5

Complex tasks often exceed the capacity of any single agent. A chain-of-agents orchestrator
coordinates a team of specialists, each with its own domain expertise and tool chest.

A robust **Cooperation Protocol** is built upon four key architectural pillars (Table 7.1):

| Protocol Element | Definition |
|:---|:---|
| **Message Format** | A shared envelope structure (e.g., JSON with sender, recipient, task_id, payload) |
| **Role Declaration** | Each agent registers its name, capabilities, and accepted task types |
| **Task Delegation Scheme** | A typed request specifying target agent, action, inputs, and expected output schema |
| **Status Signaling** | Standardized status field (pending, running, done, error) for sequencing |

This section implements a **Market Intelligence System** with three specialist agents that
contribute findings to shared **episodic memory**, coordinated by a manager agent.

> **📘 Standards Connection: MCP and A2A Protocols** *(Book p. 188)*
>
> The cooperation protocol elements (message format, role declaration, task delegation scheme, status signaling) align with emerging industry standards. The **Model Context Protocol (MCP)** and **Agent-to-Agent (A2A) protocol**, introduced in Chapter 6, formalize exactly this message schema and role-declaration layer into open, interoperable specifications.
>
> Where Chapter 6 covered their structure and negotiation mechanics, this chapter applies those foundations: the cooperation protocol described here is the practical expression of what MCP and A2A standardize at the wire level.


#### Figure 7.3 — Memory-Augmented Agent Architecture *(Book p. 189)*

```
  External Inputs ──▶ Perception ──▶ Agent Core (Cognition & Reasoning)
                                         │              │
                               Active Context      Memory Query
                               Manager   │              │
                                  ▼      ▼              ▼
                        ┌─────────────┐  ┌──────────────────────┐
                        │   Working    │  │  Vector Databases    │◀── External
                        │   Memory     │  │  & Retrieval         │    DBs/APIs
                        │ (Current     │  └──────────┬───────────┘
                        │  Prompt)     │             │
                        └──────┬───────┘    ┌────────┼────────┐
                               │           ▼                 ▼
                    Store Session    ┌──────────────┐  ┌──────────────┐
                    Context/Events   │  Episodic    │  │  Semantic    │
                               │    │  Memory      │  │  Memory      │
                               └───▶│ (Historical  │  │ (Facts &     │
                                    │  Interactions)│  │  Knowledge)  │◀── Ingestion
                                    └──────────────┘  └──────────────┘
```

**Working memory** holds the immediate context (scratchpad). **Episodic memory** stores historical interaction logs. **Semantic memory** contains durable factual knowledge. The Agent Core queries long-term memory and loads relevant context into working memory for each task.


In [4]:
# =============================================================================
# Section 4: Specialist Agents — NewsAgent, FinancialAgent, SentimentAgent
# Ref: Section 7.5, pp. 190–191 — "Implementation Example: Market Intelligence"
#
# Each specialist accepts a company name and returns a structured payload.
# All three wrap results in the same dict layout:
#   {"source": ..., "status": ..., "data": ...}
# making outputs predictable for the orchestrator.
#
# Mock data is drawn directly from the chapter's own code listings.
# In production, uncomment the API calls shown in comments.
# =============================================================================

@graceful_fallback(
    fallback_return={"source": "NewsAgent", "status": "error", "data": []},
    section="7.5",
)
def NewsAgent(company_name: str) -> Dict[str, Any]:
    """Fetches top news headlines from a data source."""
    log_step("7.5", 1, f"[NewsAgent] Fetching headlines for '{company_name}'...")
    headlines = [
        f"{company_name}: Q3 earnings beat analyst estimates by 8%",
        f"{company_name} announces expansion into cloud infrastructure",
        f"Regulatory review concluded; {company_name} cleared to proceed",
    ]
    # In production, uncomment: response = news_api.get(company_name)
    response = {"data": headlines}
    return {"source": "NewsAgent", "status": "success", "data": response["data"]}


@graceful_fallback(
    fallback_return={"source": "FinancialAgent", "status": "error", "data": {}},
    section="7.5",
)
def FinancialAgent(company_name: str) -> Dict[str, Any]:
    """Fetches financial data from a data source."""
    log_step("7.5", 2, f"[FinancialAgent] Fetching financial data for '{company_name}'...")
    response = {
        "data": {
            "pe_ratio": 24.5,
            "revenue_growth": 0.12,
            "debt_to_equity": 0.38,
            "last_close": 142.73,
        }
    }
    # In production, uncomment: response = financial_api.get(company_name)
    return {"source": "FinancialAgent", "status": "success", "data": response["data"]}


@graceful_fallback(
    fallback_return={"source": "SentimentAgent", "status": "error", "data": {}},
    section="7.5",
)
def SentimentAgent(company_name: str) -> Dict[str, Any]:
    """Fetches a sentiment score from a data source."""
    log_step("7.5", 3, f"[SentimentAgent] Fetching sentiment for '{company_name}'...")
    response = {
        "data": {
            "score": 0.72,
            "label": "positive",
            "evidence": [
                f"Investor confidence in {company_name} remains high.",
                "Social media tone broadly favorable this week.",
            ],
        }
    }
    # In production, uncomment: response = sentiment_api.get(company_name)
    return {"source": "SentimentAgent", "status": "success", "data": response["data"]}


log_success("Specialist agents defined: NewsAgent, FinancialAgent, SentimentAgent.")

[SUCCESS] Specialist agents defined: NewsAgent, FinancialAgent, SentimentAgent.


In [5]:
# =============================================================================
# Section 4: Manager Agent with Episodic Memory
# Ref: Section 7.4, pp. 187–189 — Cooperation Protocol
# Ref: Section 7.5, pp. 189–193 — Memory-Augmented Multi-Agent Systems
#
# The manager agent coordinates the specialists, stores results in episodic
# memory (timestamped interaction log), and prepares context for synthesis.
# =============================================================================

from datetime import datetime


class ManagerAgent:
    """Orchestrates specialist agents and maintains shared episodic memory.

    Implements the Cooperation Protocol from Section 7.4:
    - Role Declaration: each agent has a defined role and output schema
    - Task Delegation: manager dispatches to each specialist in turn
    - Status Signaling: checks 'status' field in agent responses
    - Shared Memory: episodic_memory stores timestamped interaction records
    """

    def __init__(self, company_name: str):
        self.company_name = company_name
        self.episodic_memory: List[Dict[str, Any]] = []  # Section 7.5 — episodic memory
        self.results: Dict[str, Any] = {}

    def delegate_and_collect(self) -> Dict[str, Any]:
        """Dispatch tasks to each specialist and store results in episodic memory."""
        log_info("=" * 60)
        log_step("7.4", 1, f"ManagerAgent dispatching tasks for '{self.company_name}'...")
        log_info("=" * 60)

        specialists = {
            "NewsAgent": NewsAgent,
            "FinancialAgent": FinancialAgent,
            "SentimentAgent": SentimentAgent,
        }

        for agent_name, agent_func in specialists.items():
            result = agent_func(self.company_name)
            self.results[agent_name] = result

            # Write to episodic memory — Section 7.5
            memory_entry = {
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "agent": agent_name,
                "status": result.get("status", "unknown"),
                "data_summary": str(result.get("data", ""))[:120],
            }
            self.episodic_memory.append(memory_entry)
            log_info(f"  Episodic memory updated: {agent_name} -> {result['status']}")

        log_success(f"All specialists reported. {len(self.episodic_memory)} memory entries.")
        return self.results

    def show_episodic_memory(self):
        """Display the episodic memory log for inspection."""
        log_info("Episodic Memory Log:")
        for entry in self.episodic_memory:
            log_info(
                f"  [{entry['timestamp']}] {entry['agent']}: "
                f"status={entry['status']}, "
                f"data={entry['data_summary'][:80]}..."
            )


log_success("ManagerAgent class defined with episodic memory.")

[SUCCESS] ManagerAgent class defined with episodic memory.


In [6]:
# =============================================================================
# Section 4: Demo — Market Intelligence System
# Ref: Sections 7.4-7.5 — Manager delegates to 3 specialists
# =============================================================================

log_info("=" * 60)
log_info("Section 4 Demo: Market Intelligence System for 'TechCorp'")
log_info("=" * 60)

manager = ManagerAgent("TechCorp")
results = manager.delegate_and_collect()

print()
manager.show_episodic_memory()

print()
log_info("Raw specialist results:")
for agent_name, result in results.items():
    print(f"  {agent_name}: {json.dumps(result, indent=2, default=str)}")

[INFO] ============================================================
[INFO] Section 4 Demo: Market Intelligence System for 'TechCorp'
[INFO] ============================================================
[INFO] ============================================================
[Section 7.4 | Step 1] ManagerAgent dispatching tasks for 'TechCorp'...
[INFO] ============================================================
[Section 7.5 | Step 1] [NewsAgent] Fetching headlines for 'TechCorp'...
[SUCCESS] NewsAgent completed. [Section 7.5]
[INFO]   Episodic memory updated: NewsAgent -> success
[Section 7.5 | Step 2] [FinancialAgent] Fetching financial data for 'TechCorp'...
[SUCCESS] FinancialAgent completed. [Section 7.5]
[INFO]   Episodic memory updated: FinancialAgent -> success
[Section 7.5 | Step 3] [SentimentAgent] Fetching sentiment for 'TechCorp'...
[SUCCESS] SentimentAgent completed. [Section 7.5]
[INFO]   Episodic memory updated: SentimentAgent -> success
[SUCCESS] All specialists reported. 3 me

---
## Section 5: Conflict Resolution Mechanisms

**Chapter Reference:** Section 7.6

When multiple specialists analyze complex information, disagreements are inevitable.
The architectural strength of a multi-agent system is measured by its capacity to
**resolve conflict productively** (Figure 7.4).

The arbitration workflow consists of four stages:
1. **Conflict Detection** — Semantic similarity or numerical divergence check
2. **Automated Arbitration** — Arbiter agent consults a knowledge base
3. **Confidence-Based Consensus** — Accept if confidence exceeds threshold (e.g., 95%)
4. **Human Escalation** — Route to human reviewer if confidence is low

This section extends the `ManagerAgent` with a `synthesize_report` method that computes
a `conflict_score` to detect divergence between sentiment and financial signals.

In [7]:
# =============================================================================
# Section 5: Conflict Resolution — synthesize_report with conflict_score
# Ref: Section 7.6, pp. 193–194 — "Implementation Example"
#
# conflict_score = abs(sentiment_score - (stock_change / 10))
#
# sentiment_score is bounded [-1, 1] by the SentimentAgent.
# stock_change is expressed as a percentage (e.g., 5.0 for 5%).
# Dividing by 10 maps a typical daily swing of +/-10% onto the same [-1, 1] scale.
# The 0.5 threshold represents a half-unit divergence on this normalized scale.
# =============================================================================

def synthesize_report(
    results: Dict[str, Any],
    stock_change_pct: float = 5.0,
) -> str:
    """Aggregate specialist findings into a single report with conflict detection.

    Parameters
    ----------
    results : dict
        Output from ManagerAgent.delegate_and_collect().
    stock_change_pct : float
        Simulated stock price change percentage for conflict detection.
        Default 5.0 represents a 5% positive move — aligned with chapter example.
    """
    log_step("7.6", 1, "Synthesizing market intelligence report...")

    financial_data = results.get("FinancialAgent", {}).get("data", {})
    sentiment_data = results.get("SentimentAgent", {}).get("data", {})
    news_items = results.get("NewsAgent", {}).get("data", [])

    # Extract key values
    sentiment_score = sentiment_data.get("score", 0.0)

    # Conflict score formula from Section 7.6, p. 27
    # Normalizes both inputs to a compatible scale before comparing
    conflict_score = abs(sentiment_score - (stock_change_pct / 10))

    # Build the report
    headlines = "; ".join(news_items[:2]) if news_items else "no headlines"

    report = "Market Intelligence Report\n"
    report += "=" * 40 + "\n"
    report += f"Recent news: {headlines}\n"
    report += (
        f"P/E ratio: {financial_data.get('pe_ratio', 'N/A')}, "
        f"Revenue growth: {financial_data.get('revenue_growth', 0):.0%}\n"
    )

    # Conflict detection — threshold 0.5 (Section 7.6, p. 27)
    if conflict_score > 0.5:
        report += (
            f"\n**Conflict Detected (Score: {conflict_score:.2f}):** "
            f"A significant discrepancy exists between public sentiment "
            f"(score={sentiment_score}) and market performance "
            f"(change={stock_change_pct}%). Further investigation recommended."
        )
        log_warning(
            f"Conflict detected! Score: {conflict_score:.2f} > 0.5 threshold. "
            f"Sentiment={sentiment_score}, StockChange={stock_change_pct}%"
        )
    else:
        report += (
            f"\n**Alignment Confirmed (Score: {conflict_score:.2f}):** "
            f"Public perception and market performance are well-aligned. "
            f"No material discrepancy detected."
        )
        log_success(
            f"Alignment confirmed. Conflict score: {conflict_score:.2f} "
            f"(within 0.5 threshold)."
        )

    return report


log_success("synthesize_report function defined with conflict_score detection.")

[SUCCESS] synthesize_report function defined with conflict_score detection.


In [8]:
# =============================================================================
# Section 5: Demo — Conflict Resolution in Action
# Ref: Section 7.6 — Two scenarios: aligned signals vs. conflicting signals
# =============================================================================

log_info("=" * 60)
log_info("Section 5 Demo: Conflict Resolution")
log_info("=" * 60)

# --- Scenario A: Aligned signals ---
# sentiment_score=0.72, stock_change=5.0% -> conflict_score = |0.72 - 0.5| = 0.22
log_info("Scenario A: Aligned signals (stock +5%, sentiment 0.72)")
report_aligned = synthesize_report(results, stock_change_pct=5.0)
print()
print(report_aligned)
print()

# --- Scenario B: Conflicting signals ---
# sentiment_score=0.72, stock_change=-8.0% -> conflict_score = |0.72 - (-0.8)| = 1.52
log_info("Scenario B: Conflicting signals (stock -8%, sentiment 0.72)")
report_conflict = synthesize_report(results, stock_change_pct=-8.0)
print()
print(report_conflict)

log_success("Section 5 complete. Both conflict scenarios demonstrated.")

[INFO] ============================================================
[INFO] Section 5 Demo: Conflict Resolution
[INFO] ============================================================
[INFO] Scenario A: Aligned signals (stock +5%, sentiment 0.72)
[Section 7.6 | Step 1] Synthesizing market intelligence report...
[SUCCESS] Alignment confirmed. Conflict score: 0.22 (within 0.5 threshold).

Market Intelligence Report
Recent news: TechCorp: Q3 earnings beat analyst estimates by 8%; TechCorp announces expansion into cloud infrastructure
P/E ratio: 24.5, Revenue growth: 12%

**Alignment Confirmed (Score: 0.22):** Public perception and market performance are well-aligned. No material discrepancy detected.

[INFO] Scenario B: Conflicting signals (stock -8%, sentiment 0.72)
[Section 7.6 | Step 1] Synthesizing market intelligence report...
[WARNING] Conflict detected! Score: 1.52 > 0.5 threshold. Sentiment=0.72, StockChange=-8.0%

Market Intelligence Report
Recent news: TechCorp: Q3 earnings beat anal

---
## Section 7: Agentic Workflow — Insurance Claims Processing

**Chapter Reference:** Section 7.7b

This section demonstrates a more complex, multi-agent workflow modeled as a **state machine**
(Figure 7.5). Five specialized agents handle different stages of insurance claims processing:

| Agent | Role |
|:---|:---|
| **Intake Agent** | OCR/NLP to digitize claim forms |
| **Validator Agent** | Checks policy status, fraud signals, required documentation |
| **Classifier Agent** | Assesses claim type, urgency, and risk level |
| **Payout Agent** | Calculates settlement and processes payment |
| **Escalation Agent** | Routes high-risk or low-confidence claims to human review |

**Guard Conditions** (Table 7.2):

| Guard | Action |
|:---|:---|
| `claim_amount > threshold` | Route to human for approval |
| Validation fails | Transition to Closed: Rejected |
| `confidence_score < 0.85` | Escalate to Pending Human Review |

Three test claims:

| Claim ID | Amount | Type | Expected Path |
|:---|:---|:---|:---|
| CLM-4821 | $8,400 | water_damage | Auto-approved (confidence 0.91) |
| CLM-5099 | $47,000 | fire_damage | Escalated (confidence 0.79) |
| CLM-5100 | $3,200 | theft | Rejected at validation (expired policy) |

#### Figure 7.5 — Insurance Claim State Machine *(Book p. 199)*

```
  ● ──▶ [Intake] ──Form Submitted──▶ [Validating] ──Valid──▶ [Assessing Risk]
                                         │                        │         │
                                      Invalid               Low Risk   High Risk
                                         │                        │         │
                                         ▼                        ▼         ▼
                                  [Closed:               [Processing  [Pending Human
                                   Rejected]              Payout]      Review]
                                      ▲                  │    │          │      │
                                      │            Success│ Payment  Approved  Rejected
                                      │                  │  Failed     │      │
                                      │                  ▼    │        ▼      │
                            Rejected  │           [Closed:    └──▶[Processing │
                            (Human)───┘            Approved]       Payout]    │
                                  ▲                                           │
                                  └───────────────────────────────────────────┘
```

**Guard conditions** (Table 7.2): `claim_amount > threshold` → human approval; `validation fails` → rejected; `confidence_score < 0.85` → escalate to human review. At each transition, the system records the agent's reasoning, tools used, and human decisions for a complete audit trail.


In [9]:
# =============================================================================
# Section 7: Insurance Claims — Test Data and Policy Database
# Ref: Section 7.7b, pp. 198–200 — Multi-Agent Insurance Claims Workflow
# Ref: Strategy §6.4 — Test claims
# =============================================================================

# --- Policy Database (simulated) ---
policy_db = {
    "POL-992317": {"status": "active", "holder": "Alice Johnson", "fraud_flag": False},
    "POL-110482": {"status": "active", "holder": "Bob Martinez", "fraud_flag": False},
    "POL-EXPIRED": {"status": "expired", "holder": "Charlie Lee", "fraud_flag": False},
}

# --- Test Claims (Strategy §6.4) ---
test_claims = [
    {
        "claim_id": "CLM-4821",
        "policy_id": "POL-992317",
        "claim_type": "water_damage",
        "amount": 8400.00,
        "description": "Burst pipe caused flooding in basement. Damage to walls and flooring.",
    },
    {
        "claim_id": "CLM-5099",
        "policy_id": "POL-110482",
        "claim_type": "fire_damage",
        "amount": 47000.00,
        "description": "Kitchen fire spread to living area. Significant structural damage.",
    },
    {
        "claim_id": "CLM-5100",
        "policy_id": "POL-EXPIRED",
        "claim_type": "theft",
        "amount": 3200.00,
        "description": "Laptop and electronics stolen during home break-in.",
    },
]

log_info(f"Policy database: {len(policy_db)} policies loaded.")
log_info(f"Test claims: {len(test_claims)} claims prepared.")

[INFO] Policy database: 3 policies loaded.
[INFO] Test claims: 3 claims prepared.


In [10]:
# =============================================================================
# Section 7: Insurance Claims — Five Specialized Agents
# Ref: Section 7.7b, pp. 198–199 — Agent roles
# =============================================================================

@graceful_fallback(fallback_return=None, section="7.7b")
def intake_agent(claim: Dict[str, Any]) -> Dict[str, Any]:
    """Intake Agent: Digitize and extract fields from claim submission.
    In production, this would use OCR and NLP on submitted forms."""
    claim_id = claim.get("claim_id", "N/A")
    log_step("7.7b", 1, f"[Intake] Processing claim {claim_id}...")

    # Simulate OCR/NLP field extraction
    extracted = {
        "claim_id": claim["claim_id"],
        "policy_id": claim["policy_id"],
        "claim_type": claim["claim_type"],
        "amount": claim["amount"],
        "description": claim["description"],
        "status": "intake_complete",
    }
    log_info(f"  Extracted: type={extracted['claim_type']}, amount=${extracted['amount']:,.2f}")
    return extracted


@graceful_fallback(fallback_return=False, section="7.7b")
def validator_agent(claim_record: Dict[str, Any], policies: Dict) -> bool:
    """Validator Agent: Check policy status and fraud signals.
    Guard: validation fails -> Closed: Rejected."""
    claim_id = claim_record.get("claim_id", "N/A")
    policy_id = claim_record.get("policy_id", "N/A")
    log_step("7.7b", 2, f"[Validator] Validating claim {claim_id}, policy {policy_id}...")

    policy = policies.get(policy_id)
    if policy is None:
        log_error(f"  Policy {policy_id} not found in database.")
        return False

    if policy["status"] != "active":
        log_error(f"  Policy {policy_id} status: {policy['status']}. Claim rejected.")
        return False

    if policy.get("fraud_flag", False):
        log_warning(f"  Fraud flag detected on policy {policy_id}.")
        return False

    log_info(f"  Policy {policy_id} validated: active, no fraud flag.")
    return True


@graceful_fallback(
    fallback_return={"confidence_score": 0.5, "risk": "high"},
    section="7.7b",
)
def classifier_agent(claim_record: Dict[str, Any]) -> Dict[str, Any]:
    """Classifier Agent: Assess claim type, urgency, and risk level.
    Guard: confidence_score < 0.85 -> escalate to Pending Human Review."""
    claim_id = claim_record.get("claim_id", "N/A")
    claim_type = claim_record.get("claim_type", "unknown")
    amount = claim_record.get("amount", 0.0)
    log_step("7.7b", 3, f"[Classifier] Assessing risk for claim {claim_id}...")

    # Rule-based classification (Simulation Mode)
    # High-value claims or fire/explosion types get lower confidence
    if amount > 25000 or claim_type in ["fire_damage", "explosion"]:
        classification = {
            "confidence_score": 0.79,
            "risk": "high",
            "claim_type": claim_type,
        }
    else:
        classification = {
            "confidence_score": 0.91,
            "risk": "low",
            "claim_type": claim_type,
        }

    log_info(
        f"  Classification: confidence={classification['confidence_score']}, "
        f"risk={classification['risk']}"
    )
    return classification


@graceful_fallback(fallback_return=False, section="7.7b")
def payout_agent(claim_record: Dict[str, Any]) -> bool:
    """Payout Agent: Calculate settlement and process payment."""
    claim_id = claim_record.get("claim_id", "N/A")
    amount = claim_record.get("amount", 0.0)
    log_step("7.7b", 5, f"[Payout] Processing settlement of ${amount:,.2f} for {claim_id}...")
    time.sleep(0.5)  # Simulate payment processing
    log_success(f"  Settlement of ${amount:,.2f} paid for claim {claim_id}.")
    return True


log_success("Insurance claims agents defined: intake, validator, classifier, payout.")

[SUCCESS] Insurance claims agents defined: intake, validator, classifier, payout.


In [11]:
# =============================================================================
# Section 7: Claims Workflow Manager — State Machine
# Ref: Section 7.7b, pp. 199–200 — Figure 7.5 and Table 7.2
#
# State transitions:
#   Intake -> Validating -> Assessing Risk -> Processing Payout -> Closed: Approved
#                |                 |                    |
#                v                 v                    v
#         Closed: Rejected   Pending Human Review  Closed: Rejected
#                             (approve -> Payout)    (payment failed)
#                             (reject  -> Rejected)
#
# Guard: confidence_score < 0.85 -> escalate to Pending Human Review
# =============================================================================

def claims_workflow_manager(
    claim: Dict[str, Any],
    policies: Dict,
) -> Dict[str, Any]:
    """Process a single insurance claim through the state machine.

    Returns an audit record with all state transitions and decisions.
    """
    claim_id = claim.get("claim_id", "N/A")
    audit_trail: List[Dict[str, str]] = []

    log_info("=" * 60)
    log_info(f"Claims Workflow: Processing {claim_id}")
    log_info("=" * 60)

    def log_transition(from_state: str, to_state: str, reason: str):
        entry = {"from": from_state, "to": to_state, "reason": reason}
        audit_trail.append(entry)
        log_info(f"  State: {from_state} -> {to_state} ({reason})")

    # --- State: Intake ---
    claim_record = intake_agent(claim)
    if claim_record is None:
        log_transition("Intake", "Closed: Rejected", "Intake agent failed")
        return {"claim_id": claim_id, "final_state": "Closed: Rejected", "audit": audit_trail}
    log_transition("Start", "Intake", "Form submitted")

    # --- State: Validating ---
    is_valid = validator_agent(claim_record, policies)
    if not is_valid:
        log_transition("Intake", "Closed: Rejected", "Validation failed")
        log_error(f"Claim {claim_id} REJECTED at validation.")
        return {"claim_id": claim_id, "final_state": "Closed: Rejected", "audit": audit_trail}
    log_transition("Intake", "Validating", "All fields populated")
    log_transition("Validating", "Assessing Risk", "Validation passed")

    # --- State: Assessing Risk ---
    classification = classifier_agent(claim_record)
    confidence = classification.get("confidence_score", 0.0)
    risk = classification.get("risk", "high")

    # Guard: confidence_score < 0.85 -> escalate (Table 7.2)
    if confidence < 0.85:
        log_transition("Assessing Risk", "Pending Human Review",
                       f"confidence_score {confidence} < 0.85 threshold")
        log_warning(f"Claim {claim_id} escalated for human review (confidence={confidence}).")

        # HITL: Escalation Agent
        log_step("7.7b", 4, f"[Escalation] Claim {claim_id} awaiting human decision...")
        if LIVE_MODE and INTERACTIVE_MODE:
            decision = ""
            while decision not in ["approve", "reject"]:
                decision = input("  Type 'approve' or 'reject': ").lower().strip()
        else:
            log_mock("  Auto-approving after 2s delay (Simulation Mode)...")
            time.sleep(2)
            decision = "approve"

        if decision == "reject":
            log_transition("Pending Human Review", "Closed: Rejected", "Human rejected")
            log_error(f"Claim {claim_id} REJECTED by human reviewer.")
            return {"claim_id": claim_id, "final_state": "Closed: Rejected", "audit": audit_trail}
        else:
            log_transition("Pending Human Review", "Processing Payout", "Human approved")
            log_success(f"Claim {claim_id} APPROVED by human reviewer.")
    else:
        log_transition("Assessing Risk", "Processing Payout",
                       f"Low risk, confidence {confidence} >= 0.85")

    # --- State: Processing Payout ---
    payment_ok = payout_agent(claim_record)
    if not payment_ok:
        log_transition("Processing Payout", "Closed: Rejected", "Payment failed")
        log_error(f"Claim {claim_id} REJECTED — payment failure.")
        return {"claim_id": claim_id, "final_state": "Closed: Rejected", "audit": audit_trail}

    log_transition("Processing Payout", "Closed: Approved", "Settlement paid")
    log_success(f"Claim {claim_id} APPROVED and settled.")

    return {"claim_id": claim_id, "final_state": "Closed: Approved", "audit": audit_trail}


log_success("claims_workflow_manager defined — full state machine with guard conditions.")

[SUCCESS] claims_workflow_manager defined — full state machine with guard conditions.


In [12]:
# =============================================================================
# Section 7: Demo — Run All 3 Test Claims
# Ref: Strategy §6.4 — Expected outcomes:
#   CLM-4821: Auto-approved (confidence 0.91, water_damage, $8,400)
#   CLM-5099: Escalated then auto-approved (confidence 0.79, fire_damage, $47,000)
#   CLM-5100: Rejected at validation (expired policy)
# =============================================================================

log_info("=" * 60)
log_info("Section 7 Demo: Insurance Claims Workflow — 3 Test Claims")
log_info("=" * 60)

claim_results = {}
for claim in test_claims:
    result = claims_workflow_manager(claim, policy_db)
    claim_results[result["claim_id"]] = result
    print()

# --- Summary and Audit Trail ---
log_info("=" * 60)
log_info("Claims Processing Summary:")
for cid, result in claim_results.items():
    final = result["final_state"]
    badge_func = log_success if "Approved" in final else log_error
    badge_func(f"  {cid}: {final}")

print()
log_info("Audit Trails:")
for cid, result in claim_results.items():
    log_info(f"  {cid}:")
    for entry in result["audit"]:
        log_info(f"    {entry['from']} -> {entry['to']} ({entry['reason']})")

log_success("Section 7 complete. All 3 test claims processed with full audit trails.")

[INFO] ============================================================
[INFO] Section 7 Demo: Insurance Claims Workflow — 3 Test Claims
[INFO] ============================================================
[INFO] ============================================================
[INFO] Claims Workflow: Processing CLM-4821
[INFO] ============================================================
[Section 7.7b | Step 1] [Intake] Processing claim CLM-4821...
[INFO]   Extracted: type=water_damage, amount=$8,400.00
[SUCCESS] intake_agent completed. [Section 7.7b]
[INFO]   State: Start -> Intake (Form submitted)
[Section 7.7b | Step 2] [Validator] Validating claim CLM-4821, policy POL-992317...
[INFO]   Policy POL-992317 validated: active, no fraud flag.
[SUCCESS] validator_agent completed. [Section 7.7b]
[INFO]   State: Intake -> Validating (All fields populated)
[INFO]   State: Validating -> Assessing Risk (Validation passed)
[Section 7.7b | Step 3] [Classifier] Assessing risk for claim CLM-4821...
[INFO] 

[SUCCESS]   Settlement of $8,400.00 paid for claim CLM-4821.
[SUCCESS] payout_agent completed. [Section 7.7b]
[INFO]   State: Processing Payout -> Closed: Approved (Settlement paid)
[SUCCESS] Claim CLM-4821 APPROVED and settled.

[INFO] ============================================================
[INFO] Claims Workflow: Processing CLM-5099
[INFO] ============================================================
[Section 7.7b | Step 1] [Intake] Processing claim CLM-5099...
[INFO]   Extracted: type=fire_damage, amount=$47,000.00
[SUCCESS] intake_agent completed. [Section 7.7b]
[INFO]   State: Start -> Intake (Form submitted)
[Section 7.7b | Step 2] [Validator] Validating claim CLM-5099, policy POL-110482...
[INFO]   Policy POL-110482 validated: active, no fraud flag.
[SUCCESS] validator_agent completed. [Section 7.7b]
[INFO]   State: Intake -> Validating (All fields populated)
[INFO]   State: Validating -> Assessing Risk (Validation passed)
[Section 7.7b | Step 3] [Classifier] Assessing risk 

[INFO]   State: Pending Human Review -> Processing Payout (Human approved)
[SUCCESS] Claim CLM-5099 APPROVED by human reviewer.
[Section 7.7b | Step 5] [Payout] Processing settlement of $47,000.00 for CLM-5099...


[SUCCESS]   Settlement of $47,000.00 paid for claim CLM-5099.
[SUCCESS] payout_agent completed. [Section 7.7b]
[INFO]   State: Processing Payout -> Closed: Approved (Settlement paid)
[SUCCESS] Claim CLM-5099 APPROVED and settled.

[INFO] ============================================================
[INFO] Claims Workflow: Processing CLM-5100
[INFO] ============================================================
[Section 7.7b | Step 1] [Intake] Processing claim CLM-5100...
[INFO]   Extracted: type=theft, amount=$3,200.00
[SUCCESS] intake_agent completed. [Section 7.7b]
[INFO]   State: Start -> Intake (Form submitted)
[Section 7.7b | Step 2] [Validator] Validating claim CLM-5100, policy POL-EXPIRED...
[ERROR]   Policy POL-EXPIRED status: expired. Claim rejected.
[SUCCESS] validator_agent completed. [Section 7.7b]
[INFO]   State: Intake -> Closed: Rejected (Validation failed)
[ERROR] Claim CLM-5100 REJECTED at validation.

[INFO] ============================================================
[I

---
## Summary and Key Takeaways

- A **Chain-of-Agents Orchestrator** coordinates specialist agents through the four pillars of the Cooperation Protocol: message format, role declaration, task delegation scheme, and status signaling (§7.4).
- **Episodic memory** preserves timestamped interaction records across agent handoffs, making multi-agent execution inspectable and auditable (§7.5).
- **Conflict resolution** quantifies disagreement between agent outputs with a calibrated `conflict_score` and escalates when divergence crosses the threshold (§7.6).
- **Agentic workflows** model business processes as state machines: guard conditions govern transitions, human-in-the-loop gates handle low-confidence cases, and every transition is recorded in an audit trail (§7.7b).
- **Fail-gracefully architecture** — `@graceful_fallback` on every agent function and `MockLLM` in Simulation Mode — keeps the entire pipeline running with or without an API key.

**Further reading:** Chapter 7 of *30 Agents Every AI Engineer Must Build* by Imran Ahmad (Packt, 2026) — Sections 7.4–7.6 and 7.7b, pp. 186–194, 198–201.
